In [1]:
import pymssql

# Criar o banco ecommerce
conn = pymssql.connect(
    server='localhost',
    user='sa',
    password='Senha@1234',
    database='master',
    autocommit=True
)
cursor = conn.cursor()
cursor.execute("IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'ecommerce') CREATE DATABASE ecommerce")
conn.close()
print("Banco criado!")

# Conectar ao banco ecommerce e criar tabelas
conn = pymssql.connect(
    server='localhost',
    user='sa',
    password='Senha@1234',
    database='ecommerce'
)
cursor = conn.cursor()

cursor.execute('''
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='clientes')
    CREATE TABLE clientes (
        id_cliente INT PRIMARY KEY,
        nome VARCHAR(100),
        email VARCHAR(100),
        cidade VARCHAR(50)
    )
''')

cursor.execute('''
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='produtos')
    CREATE TABLE produtos (
        id_produto INT PRIMARY KEY,
        nome VARCHAR(100),
        categoria VARCHAR(50),
        preco FLOAT
    )
''')

cursor.execute('''
    IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='pedidos')
    CREATE TABLE pedidos (
        id_pedido INT PRIMARY KEY,
        id_cliente INT,
        id_produto INT,
        quantidade INT,
        valor_total FLOAT,
        status VARCHAR(20),
        data_pedido DATE
    )
''')

conn.commit()
print("Tabelas criadas com sucesso!")

Banco criado!
Tabelas criadas com sucesso!


In [2]:
# Limpar e reinserir dados
cursor.execute("DELETE FROM pedidos")
cursor.execute("DELETE FROM clientes")
cursor.execute("DELETE FROM produtos")

cursor.execute("INSERT INTO clientes VALUES (1, 'Ana Lima', 'ana@email.com', 'São Paulo')")
cursor.execute("INSERT INTO clientes VALUES (2, 'João Silva', 'joao@email.com', 'Curitiba')")
cursor.execute("INSERT INTO clientes VALUES (3, 'Maria Souza', 'maria@email.com', 'Florianópolis')")

cursor.execute("INSERT INTO produtos VALUES (1, 'Notebook', 'Eletrônicos', 3500.00)")
cursor.execute("INSERT INTO produtos VALUES (2, 'Mouse', 'Periféricos', 150.00)")
cursor.execute("INSERT INTO produtos VALUES (3, 'Teclado', 'Periféricos', 200.00)")

cursor.execute("INSERT INTO pedidos VALUES (1, 1, 1, 1, 3500.00, 'pendente', '2024-01-10')")
cursor.execute("INSERT INTO pedidos VALUES (2, 2, 2, 2, 300.00, 'enviado', '2024-01-11')")
cursor.execute("INSERT INTO pedidos VALUES (3, 3, 3, 1, 200.00, 'entregue', '2024-01-12')")

conn.commit()
print("Dados inseridos com sucesso!")

Dados inseridos com sucesso!


In [3]:
import pandas as pd
from minio import Minio
import io

# Conectar ao MinIO
minio_client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

# Extrair cada tabela e salvar no MinIO como CSV
tabelas = ['clientes', 'produtos', 'pedidos']

for tabela in tabelas:
    # Ler tabela do SQL Server
    df = pd.read_sql(f"SELECT * FROM {tabela}", conn)
    print(f"Tabela {tabela}: {len(df)} registros")
    
    # Converter para CSV em memória
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False)
    csv_buffer.seek(0)
    
    # Salvar no MinIO bucket landing-zone
    minio_client.put_object(
        "landing-zone",
        f"{tabela}.csv",
        csv_buffer,
        length=csv_buffer.getbuffer().nbytes,
        content_type="text/csv"
    )
    print(f"{tabela}.csv salvo no MinIO landing-zone!")

print("\nExtração concluída!")


Tabela clientes: 3 registros
clientes.csv salvo no MinIO landing-zone!
Tabela produtos: 3 registros
produtos.csv salvo no MinIO landing-zone!
Tabela pedidos: 3 registros
pedidos.csv salvo no MinIO landing-zone!

Extração concluída!


/tmp/ipykernel_3685/1446418937.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {tabela}", conn)
/tmp/ipykernel_3685/1446418937.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {tabela}", conn)
/tmp/ipykernel_3685/1446418937.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {tabela}", conn)


In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(
    SparkSession.builder.appName("SparkMinioSQL")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
).getOrCreate()

# Configurar via SparkContext também
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://localhost:9000")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.access.key", "minioadmin")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "minioadmin")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

print("Spark iniciado!")

# Teste
df = spark.read.option("header", "true").csv("s3a://landing-zone/clientes.csv")
df.show()

Spark iniciado!
+----------+-----------+---------------+-------------+
|id_cliente|       nome|          email|       cidade|
+----------+-----------+---------------+-------------+
|         1|   Ana Lima|  ana@email.com|    São Paulo|
|         2| João Silva| joao@email.com|     Curitiba|
|         3|Maria Souza|maria@email.com|Florianópolis|
+----------+-----------+---------------+-------------+



In [3]:
tabelas = ['clientes', 'produtos', 'pedidos']

for tabela in tabelas:
    print(f"Processando {tabela}...")
    
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"s3a://landing-zone/{tabela}.csv")
    
    print(f"  {df.count()} registros lidos")
    df.show()
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"s3a://bronze/{tabela}")
    
    print(f"  {tabela} salvo como Delta Lake no bronze!")

print("\nPipeline concluído!")

Processando clientes...
  3 registros lidos
+----------+-----------+---------------+-------------+
|id_cliente|       nome|          email|       cidade|
+----------+-----------+---------------+-------------+
|         1|   Ana Lima|  ana@email.com|    São Paulo|
|         2| João Silva| joao@email.com|     Curitiba|
|         3|Maria Souza|maria@email.com|Florianópolis|
+----------+-----------+---------------+-------------+

  clientes salvo como Delta Lake no bronze!
Processando produtos...
  3 registros lidos
+----------+--------+-----------+------+
|id_produto|    nome|  categoria| preco|
+----------+--------+-----------+------+
|         1|Notebook|Eletrônicos|3500.0|
|         2|   Mouse|Periféricos| 150.0|
|         3| Teclado|Periféricos| 200.0|
+----------+--------+-----------+------+

  produtos salvo como Delta Lake no bronze!
Processando pedidos...
  3 registros lidos
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_prod

In [4]:
spark.sql("""
    INSERT INTO delta.`s3a://bronze/pedidos`
    VALUES (4, 1, 2, 3, 450.00, 'pendente', '2024-02-01')
""")
print("INSERT realizado!")
spark.read.format("delta").load("s3a://bronze/pedidos").show()

INSERT realizado!


26/05/11 22:10:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|pendente| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
|        3|         3|         3|         1|      200.0|entregue| 2024-01-12|
|        4|         1|         2|         3|      450.0|pendente| 2024-02-01|
+---------+----------+----------+----------+-----------+--------+-----------+



In [5]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "s3a://bronze/pedidos")
dt.update(
    condition="id_pedido = 1",
    set={"status": "'entregue'"}
)
print("UPDATE realizado!")
spark.read.format("delta").load("s3a://bronze/pedidos").show()

UPDATE realizado!
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
|        3|         3|         3|         1|      200.0|entregue| 2024-01-12|
|        4|         1|         2|         3|      450.0|pendente| 2024-02-01|
+---------+----------+----------+----------+-----------+--------+-----------+



In [6]:
dt.delete("id_pedido = 4")
print("DELETE realizado!")
spark.read.format("delta").load("s3a://bronze/pedidos").show()

DELETE realizado!
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
|        3|         3|         3|         1|      200.0|entregue| 2024-01-12|
+---------+----------+----------+----------+-----------+--------+-----------+



In [7]:
dt.history().select("version", "timestamp", "operation").show(truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-05-11 22:10:52|DELETE   |
|2      |2026-05-11 22:10:37|UPDATE   |
|1      |2026-05-11 22:10:20|WRITE    |
|0      |2026-05-11 22:09:57|WRITE    |
+-------+-------------------+---------+

